# Deploy IITB Junta Analytics Genie Space

This notebook deploys the exported Genie Space to your Databricks workspace.

**Prerequisites:**
- Run `python setup.py --catalog YOUR_CATALOG` from the project root to configure catalog references
- Have a SQL warehouse available in your workspace

In [ ]:
# Create widgets for parameterization
dbutils.widgets.text("warehouse_id", "", "SQL Warehouse ID")

In [ ]:
# Get widget values
warehouse_id = dbutils.widgets.get("warehouse_id")

# Validate required parameters
if not warehouse_id:
    raise ValueError("warehouse_id is required. Please set it in the widget above.")

print(f"Configuration:")
print(f"  Warehouse ID: {warehouse_id}")

In [ ]:
import json
from pathlib import Path

# Get the directory where this notebook lives
# In Databricks, use the notebook path from the context
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = str(Path(notebook_path).parent)

# Read the exported files from workspace
def read_workspace_file(filename: str) -> dict:
    """Read a JSON file from the workspace relative to this notebook."""
    file_path = f"/Workspace{notebook_dir}/{filename}"
    with open(file_path, "r") as f:
        return json.load(f)

# Load exported files
metadata = read_workspace_file("space_metadata.json")
serialized_space = read_workspace_file("serialized_space.json")

print(f"Loaded Genie Space: {metadata['title']}")
print(f"Description: {metadata['description']}")

In [ ]:
from databricks.sdk import WorkspaceClient

# Initialize Databricks client
w = WorkspaceClient()
print(f"Connected to: {w.config.host}")

# Create the Genie Space
print(f"\nCreating Genie Space: {metadata['title']}")
print(f"  Warehouse: {warehouse_id}")

space = w.genie.create_space(
    warehouse_id=warehouse_id,
    serialized_space=json.dumps(serialized_space),
    title=metadata["title"],
    description=metadata["description"],
)

print(f"\n✅ Genie Space created successfully!")
print(f"  Space ID: {space.space_id}")
print(f"  URL: {w.config.host}/genie/rooms/{space.space_id}")

In [ ]:
# Display clickable link to the Genie Space
genie_url = f"{w.config.host}/genie/rooms/{space.space_id}"
displayHTML(f'<a href="{genie_url}" target="_blank">Open Genie Space: {metadata["title"]}</a>')